# Momentum Strategy Backtest on S&P 500 (CRSP Data)

**Author:** Raffaele Perillo

A long-only cross-sectional momentum strategy on S&P 500 constituents using daily CRSP data (1992–2008), ranking stocks on 12-2 momentum and forming value-weighted decile portfolios. Rebalancing frequency (monthly/quarterly/annual) and transaction cost sensitivity (15–30 bps, calibrated on AQR’s proprietary execution cost data) are tested for robustness.

**Result:** the quarterly-rebalanced portfolio returned 13.1% annualized (Sharpe 0.46), outperforming both the S&P 500 (7.2%) and the long-short UMD factor (10.1%) over the same period.

## Index Construction


## Without Transaction Costs

In [ ]:
# @title
import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("final.csv", low_memory=False)
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "PERMNO": "permno",
    "MbrStartDt": "start",
    "MbrEndDt": "end",
    "DlyCalDt": "date",
    "DlyRet": "ret",
    "DlyCap": "me"
})

# Dates
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["start"] = pd.to_datetime(df["start"], errors="coerce")
df["end"] = pd.to_datetime(df["end"], errors="coerce")

# Numeric
df["ret"] = pd.to_numeric(df["ret"], errors="coerce")
df["me"] = pd.to_numeric(df["me"], errors="coerce")

# Basic cleaning
df = df.dropna(subset=["permno", "date", "ret", "me", "start"])
df = df[df["me"] > 0].copy()

# Sort
df = df.sort_values(["permno", "date"])

# =========================
# 1b. COUNT TRADING DAYS PER STOCK-MONTH AND FILTER (>=75%)
# =========================
df["month_end"] = df["date"].dt.to_period("M").dt.to_timestamp("M")

stock_days = (
    df.groupby(["permno", "month_end"])["date"]
    .count()
    .reset_index()
    .rename(columns={"date": "stock_days"})
)

market_days = (
    df.groupby("month_end")["date"]
    .apply(lambda x: x.dt.date.nunique())
    .reset_index()
    .rename(columns={"date": "market_days"})
)

stock_days = stock_days.merge(market_days, on="month_end", how="left")
stock_days["coverage"] = stock_days["stock_days"] / stock_days["market_days"]

valid_stock_months = stock_days[stock_days["coverage"] >= 0.75][["permno", "month_end"]]

df = df.merge(valid_stock_months, on=["permno", "month_end"], how="inner")
df = df.drop(columns=["month_end"])

# =========================
# 2. BUILD MONTHLY DATA FROM DAILY
# =========================
monthly_ret = (
    df.assign(gross=1 + df["ret"])
      .groupby(["permno", pd.Grouper(key="date", freq="ME")])["gross"]
      .prod()
      .reset_index()
)
monthly_ret["ret"] = monthly_ret["gross"] - 1
monthly_ret = monthly_ret.drop(columns=["gross"])

monthly_me = (
    df.sort_values(["permno", "date"])
      .groupby(["permno", pd.Grouper(key="date", freq="ME")])["me"]
      .agg(me_start="first", me_end="last")
      .reset_index()
)

monthly = monthly_ret.merge(monthly_me, on=["permno", "date"], how="inner")

# =========================
# 3. MOMENTUM: t-12 to t-2 (with continuity check)
# =========================
monthly = monthly.sort_values(["permno", "date"])

# Convert date to month number for continuity check
monthly["month_num"] = monthly["date"].dt.year * 12 + monthly["date"].dt.month

# Check that each row is exactly 1 calendar month after the previous row
monthly["month_diff"] = monthly.groupby("permno")["month_num"].diff()

# Count consecutive months without gaps
def count_consecutive(group):
    consecutive = pd.Series(0, index=group.index)
    count = 0
    for idx in group.index:
        if group.loc[idx, "month_diff"] == 1 or pd.isna(group.loc[idx, "month_diff"]):
            if pd.isna(group.loc[idx, "month_diff"]):
                count = 1  # first row of the group
            else:
                count += 1
        else:
            count = 1  # gap detected, reset
        consecutive.loc[idx] = count
    return consecutive

monthly["consecutive"] = monthly.groupby("permno").apply(
    count_consecutive, include_groups=False
).reset_index(level=0, drop=True)

# Calculate momentum with shift(2) and rolling(11)
monthly["mom_raw"] = (
    monthly.groupby("permno")["ret"]
    .transform(lambda x: (1 + x).shift(2).rolling(11).apply(np.prod, raw=True) - 1)
)

# Momentum is valid only if we have at least 13 consecutive months
monthly["mom"] = np.where(monthly["consecutive"] >= 13, monthly["mom_raw"], np.nan)

# Clean up temporary columns
monthly = monthly.drop(columns=["month_num", "month_diff", "consecutive", "mom_raw"])

# =========================
# 4. S&P 500 BENCHMARK (value-weighted with start-of-month cap)
# =========================
def compute_sp500_benchmark(monthly_data):
    results = []
    for dt in monthly_data["date"].unique():
        month_slice = monthly_data[monthly_data["date"] == dt].dropna(subset=["ret", "me_start"])
        total_me = month_slice["me_start"].sum()
        if total_me > 0:
            mkt_ret = (month_slice["ret"] * month_slice["me_start"] / total_me).sum()
            results.append({"date": dt, "sp500_return": mkt_ret})

    bench = pd.DataFrame(results).sort_values("date").reset_index(drop=True)
    bench["sp500_cum_return"] = (1 + bench["sp500_return"]).cumprod() - 1
    return bench

benchmark = compute_sp500_benchmark(monthly)

# =========================
# 5. STRATEGY FUNCTION (dynamic value weights, no transaction costs)
# =========================
def run_strategy(monthly_data, rebalance_type):
    dates = monthly_data["date"].drop_duplicates().sort_values()

    if rebalance_type == "1M":
        rebalance_dates = dates
    elif rebalance_type == "3M":
        rebalance_dates = dates[dates.dt.month.isin([3, 6, 9, 12])]
    elif rebalance_type == "12M":
        rebalance_dates = dates[dates.dt.month == 12]
    else:
        raise ValueError("rebalance_type must be '1M', '3M', or '12M'")

    results = []
    printed = False

    for i in range(len(rebalance_dates) - 1):
        t = rebalance_dates.iloc[i]
        t_next = rebalance_dates.iloc[i + 1]

        cross = monthly_data[
            monthly_data["date"] == t
        ].dropna(subset=["mom", "me_start"]).copy()

        if len(cross) < 20:
            continue

        # Top 10% by momentum
        n = max(1, int(np.floor(len(cross) * 0.10)))
        selected = cross.sort_values("mom", ascending=False).head(n).copy()
        selected_permnos = selected["permno"]

        if not printed:
            print(f"{'='*60}")
            print(f"FIRST VALID REBALANCE (i={i})")
            print(f"{'='*60}")
            print(f"\nRebalance date (t): {t.strftime('%Y-%m-%d')}")
            print(f"Next rebalance (t_next): {t_next.strftime('%Y-%m-%d')}")
            print(f"\n--- STEP 1: Cross-section at t ---")
            print(f"Total stocks with valid momentum: {len(cross)}")
            print(f"Top 10% = {n} stocks selected")

            print(f"\n--- STEP 2: Ranking by momentum (top 10) ---")
            print(selected[["permno", "mom", "me_start"]].head(10).to_string(index=False))
            print(f"...")
            print(f"Last selected (rank {n}):")
            print(selected[["permno", "mom", "me_start"]].tail(1).to_string(index=False))

        # Hold until next rebalance
        future = monthly_data[
            (monthly_data["date"] > t) &
            (monthly_data["date"] <= t_next) &
            (monthly_data["permno"].isin(selected_permnos))
        ].copy()

        # Compute weights each month using start-of-month cap
        for dt in future["date"].unique():
            mask = future["date"] == dt
            month_slice = future.loc[mask]
            total_me = month_slice["me_start"].sum()
            if total_me > 0:
                future.loc[mask, "weight"] = month_slice["me_start"] / total_me

        if not printed:
            print(f"\n--- STEP 3: Holding period data ---")
            print(f"Months in holding period: {future['date'].nunique()}")
            print(f"Stocks x months rows: {len(future)}")
            print(f"\nFuture dataframe (first 20 rows):")
            print(future[["permno", "date", "ret", "me_start", "weight"]].head(20).to_string(index=False))

            print(f"\n--- STEP 4: Weights for first month of holding ---")
            first_month = future[future["date"] == future["date"].min()].copy()
            first_month = first_month.sort_values("weight", ascending=False)
            print(f"Month: {first_month['date'].iloc[0].strftime('%Y-%m-%d')}")
            print(f"Total me_start: {first_month['me_start'].sum():,.2f}")
            print(f"Top 5 by weight:")
            print(first_month[["permno", "ret", "me_start", "weight"]].head().to_string(index=False))
            print(f"Sum of weights: {first_month['weight'].sum():.4f}")

        port = (future["ret"] * future["weight"]).groupby(future["date"]).sum().reset_index()
        port.columns = ["date", "portfolio_return"]

        if not printed:
            print(f"\n--- STEP 5: Portfolio returns ---")
            print(port.to_string(index=False))
            print(f"\n{'='*60}")
            printed = True

        if not port.empty:
            results.append(port)

    if len(results) == 0:
        return pd.DataFrame(columns=["date", "portfolio_return", "cum_return"])

    results = pd.concat(results, ignore_index=True).sort_values("date")
    results["cum_return"] = (1 + results["portfolio_return"]).cumprod() - 1
    return results
# =========================
# 6. SUMMARY FUNCTION
# =========================
def summarize_results(results, label):
    if results.empty:
        return {
            "rebalancing": label,
            "n_months": 0,
            "total_return": np.nan,
            "annualized_return": np.nan,
            "annualized_volatility": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan
        }

    wealth = (1 + results["portfolio_return"]).cumprod()
    total_return = wealth.iloc[-1] - 1
    n_months = len(results)
    annualized_return = wealth.iloc[-1] ** (12 / n_months) - 1
    annualized_volatility = results["portfolio_return"].std(ddof=1) * np.sqrt(12)
    rf_annualized_average = 3.738/100
    sharpe = (annualized_return - rf_annualized_average) / annualized_volatility if annualized_volatility > 0 else np.nan
    max_drawdown = (wealth / wealth.cummax() - 1).min()

    return {
        "rebalancing": label,
        "n_months": n_months,
        "total_return": total_return,
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown
    }

# =========================
# 7. RUN ALL 3 VERSIONS
# =========================
res_1m = run_strategy(monthly, "1M")
res_3m = run_strategy(monthly, "3M")
res_12m = run_strategy(monthly, "12M")

# =========================
# 7b. ALIGN ALL STRATEGIES AND BENCHMARK TO COMMON PERIOD
# =========================
start_date = max(res_1m["date"].min(), res_3m["date"].min(), res_12m["date"].min())
end_date = min(res_1m["date"].max(), res_3m["date"].max(), res_12m["date"].max())

res_1m = res_1m[(res_1m["date"] >= start_date) & (res_1m["date"] <= end_date)].copy()
res_3m = res_3m[(res_3m["date"] >= start_date) & (res_3m["date"] <= end_date)].copy()
res_12m = res_12m[(res_12m["date"] >= start_date) & (res_12m["date"] <= end_date)].copy()

res_1m["cum_return"] = (1 + res_1m["portfolio_return"]).cumprod() - 1
res_3m["cum_return"] = (1 + res_3m["portfolio_return"]).cumprod() - 1
res_12m["cum_return"] = (1 + res_12m["portfolio_return"]).cumprod() - 1

bench_common = benchmark[(benchmark["date"] >= start_date) & (benchmark["date"] <= end_date)].copy()
bench_common["sp500_cum_return"] = (1 + bench_common["sp500_return"]).cumprod() - 1

# =========================
# 7c. SUMMARY ON COMMON PERIOD
# =========================
summary = pd.DataFrame([
    summarize_results(res_1m, "1 month"),
    summarize_results(res_3m, "3 months"),
    summarize_results(res_12m, "12 months")
])

# =========================
# 8. MOMENTUM PREMIUM (portfolio - S&P 500, on common period)
# =========================
def compute_momentum_premium(strategy_results, benchmark_common, label):
    merged = strategy_results.merge(benchmark_common[["date", "sp500_return"]], on="date", how="inner")

    n = len(merged)
    if n == 0:
        return {"rebalancing": label, "premium_total": np.nan, "premium_annualized": np.nan}

    total_portfolio = (1 + merged["portfolio_return"]).cumprod().iloc[-1] - 1
    total_benchmark = (1 + merged["sp500_return"]).cumprod().iloc[-1] - 1

    premium_total = total_portfolio - total_benchmark
    premium_annualized = (1 + total_portfolio) ** (12 / n) - 1 - ((1 + total_benchmark) ** (12 / n) - 1)

    return {
        "rebalancing": label,
        "premium_total": premium_total,
        "premium_annualized": premium_annualized
    }

premium_summary = pd.DataFrame([
    compute_momentum_premium(res_1m, bench_common, "1 month"),
    compute_momentum_premium(res_3m, bench_common, "3 months"),
    compute_momentum_premium(res_12m, bench_common, "12 months")
])

# =========================
# 9. PRINT & SAVE
# =========================
print(f"=== COMMON PERIOD: {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')} ===")
print()

summary_display = summary.copy()
for col in ["total_return", "annualized_return", "annualized_volatility", "sharpe", "max_drawdown"]:
    summary_display[col] = summary_display[col].round(4)

print("=== STRATEGY PERFORMANCE (no transaction costs) ===")
print(summary_display)
print()

sp500_total = bench_common["sp500_cum_return"].iloc[-1]
sp500_n = len(bench_common)
sp500_ann = (1 + sp500_total) ** (12 / sp500_n) - 1
print("=== S&P 500 BENCHMARK (common period) ===")
print(f"Total return: {sp500_total:.4f}")
print(f"Annualized return: {sp500_ann:.4f}")
print()

premium_display = premium_summary.copy()
for col in ["premium_total", "premium_annualized"]:
    premium_display[col] = premium_display[col].round(4)
print("=== MOMENTUM PREMIUM (strategy - S&P 500) ===")
print(premium_display)

# Save
summary.to_csv("rebalancing_summary.csv", index=False)
bench_common.to_csv("sp500_benchmark.csv", index=False)
premium_summary.to_csv("momentum_premium.csv", index=False)
res_1m.to_csv("sp500_top10_momentum_1m.csv", index=False)
res_3m.to_csv("sp500_top10_momentum_3m.csv", index=False)
res_12m.to_csv("sp500_top10_momentum_12m.csv", index=False)
print(f"Start date: {start_date}")
print(f"End date: {end_date}")
print(f"N months benchmark: {len(bench_common)}")
print(f"Benchmark total return: {bench_common['sp500_cum_return'].iloc[-1]:.4f}")

FIRST VALID REBALANCE (i=12)

Rebalance date (t): 1991-12-31
Next rebalance (t_next): 1992-01-31

--- STEP 1: Cross-section at t ---
Total stocks with valid momentum: 491
Top 10% = 49 stocks selected

--- STEP 2: Ranking by momentum (top 10) ---
 permno      mom   me_start
  57592 3.147032  452208.75
  59010 1.882546 7731255.00
  61241 1.684207 1191285.75
  16432 1.646375 2838899.00
  52919 1.537078 5174292.75
  91380 1.442107 2372254.50
  18092 1.421769 2678762.50
  47896 1.354781 1985003.13
  52978 1.334419 2039323.00
  27430 1.277596 1645014.00
...
Last selected (rank 49):
 permno      mom   me_start
  65138 0.684401 7175826.63

--- STEP 3: Holding period data ---
Months in holding period: 1
Stocks x months rows: 48

Future dataframe (first 20 rows):
 permno       date       ret    me_start   weight
  10104 1992-01-31  0.275862  2057480.25 0.011019
  12052 1992-01-31  0.086119  2252931.25 0.012065
  16432 1992-01-31  0.128506  3759663.00 0.020134
  17961 1992-01-31  0.104812   62918

In [ ]:
def print_summary(results, label):
    s = summarize_results(results, label)
    print(f"{'='*45}")
    print(f"  {label.upper()} REBALANCING")
    print(f"{'='*45}")
    print(f"  Period:              {int(s['n_months'])} months")
    print(f"  Total Return:        {s['total_return']*100:.2f}%")
    print(f"  Annualized Return:   {s['annualized_return']*100:.2f}%")
    print(f"  Annualized Vol:      {s['annualized_volatility']*100:.2f}%")
    print(f"  Sharpe Ratio:        {s['sharpe']:.4f}")
    print(f"  Max Drawdown:        {s['max_drawdown']*100:.2f}%")
    print(f"{'='*45}")

# Usage:
print_summary(res_3m, "3 months")

  3 MONTHS REBALANCING
  Period:              204 months
  Total Return:        715.49%
  Annualized Return:   13.14%
  Annualized Vol:      20.27%
  Sharpe Ratio:        0.4637
  Max Drawdown:        -60.43%


In [ ]:
def print_all_summaries(res_1m, res_3m, res_12m, title="  STRATEGY PERFORMANCE - NO TRANSACTION COST "):
    print(f"{'='*65}")
    print(f"  {title}")
    print(f"{'='*65}")
    print(f"  {'':20s} {'1 MONTH':>12s} {'3 MONTHS':>12s} {'12 MONTHS':>12s}")
    print(f"{'-'*65}")

    s1 = summarize_results(res_1m, "1 month")
    s3 = summarize_results(res_3m, "3 months")
    s12 = summarize_results(res_12m, "12 months")

    print(f"  {'Period (months)':20s} {int(s1['n_months']):>12d} {int(s3['n_months']):>12d} {int(s12['n_months']):>12d}")
    print(f"  {'Total Return':20s} {s1['total_return']*100:>11.2f}% {s3['total_return']*100:>11.2f}% {s12['total_return']*100:>11.2f}%")
    print(f"  {'Annualized Return':20s} {s1['annualized_return']*100:>11.2f}% {s3['annualized_return']*100:>11.2f}% {s12['annualized_return']*100:>11.2f}%")
    print(f"  {'Annualized Vol':20s} {s1['annualized_volatility']*100:>11.2f}% {s3['annualized_volatility']*100:>11.2f}% {s12['annualized_volatility']*100:>11.2f}%")
    print(f"  {'Sharpe Ratio':20s} {s1['sharpe']:>12.4f} {s3['sharpe']:>12.4f} {s12['sharpe']:>12.4f}")
    print(f"  {'Max Drawdown':20s} {s1['max_drawdown']*100:>11.2f}% {s3['max_drawdown']*100:>11.2f}% {s12['max_drawdown']*100:>11.2f}%")
    print(f"{'='*65}")

# Usage:
print_all_summaries(res_1m, res_3m, res_12m)

    STRATEGY PERFORMANCE - NO TRANSACTION COST 
                            1 MONTH     3 MONTHS    12 MONTHS
-----------------------------------------------------------------
  Period (months)               204          204          204
  Total Return              915.53%      715.49%      474.65%
  Annualized Return          14.61%       13.14%       10.83%
  Annualized Vol             19.57%       20.27%       21.01%
  Sharpe Ratio               0.5556       0.4637       0.3378
  Max Drawdown              -42.03%      -60.43%      -66.41%


##With transaction costs

In [ ]:
import pandas as pd
import numpy as np

# =========================
# 1. LOAD DATA
# =========================
df = pd.read_csv("final.csv", low_memory=False)
df.columns = df.columns.str.strip()

df = df.rename(columns={
    "PERMNO": "permno",
    "MbrStartDt": "start",
    "MbrEndDt": "end",
    "DlyCalDt": "date",
    "DlyRet": "ret",
    "DlyCap": "me"
})

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["start"] = pd.to_datetime(df["start"], errors="coerce")
df["end"] = pd.to_datetime(df["end"], errors="coerce")
df["ret"] = pd.to_numeric(df["ret"], errors="coerce")
df["me"] = pd.to_numeric(df["me"], errors="coerce")

df = df.dropna(subset=["permno", "date", "ret", "me", "start"])
df = df[df["me"] > 0].copy()
df = df.sort_values(["permno", "date"])

# =========================
# 1b. COUNT TRADING DAYS PER STOCK-MONTH AND FILTER (>=75%)
# =========================
df["month_end"] = df["date"].dt.to_period("M").dt.to_timestamp("M")

stock_days = (
    df.groupby(["permno", "month_end"])["date"]
    .count()
    .reset_index()
    .rename(columns={"date": "stock_days"})
)

market_days = (
    df.groupby("month_end")["date"]
    .apply(lambda x: x.dt.date.nunique())
    .reset_index()
    .rename(columns={"date": "market_days"})
)

stock_days = stock_days.merge(market_days, on="month_end", how="left")
stock_days["coverage"] = stock_days["stock_days"] / stock_days["market_days"]

valid_stock_months = stock_days.loc[
    stock_days["coverage"] >= 0.75, ["permno", "month_end"]
]

df = df.merge(valid_stock_months, on=["permno", "month_end"], how="inner")
df = df.drop(columns=["month_end"])

# =========================
# 2. BUILD MONTHLY DATA FROM DAILY
# =========================
monthly_ret = (
    df.assign(gross=1 + df["ret"])
      .groupby(["permno", pd.Grouper(key="date", freq="ME")])["gross"]
      .prod()
      .reset_index()
)
monthly_ret["ret"] = monthly_ret["gross"] - 1
monthly_ret = monthly_ret.drop(columns=["gross"])

monthly_me = (
    df.sort_values(["permno", "date"])
      .groupby(["permno", pd.Grouper(key="date", freq="ME")])["me"]
      .agg(me_start="first", me_end="last")
      .reset_index()
)

monthly = monthly_ret.merge(monthly_me, on=["permno", "date"], how="inner")

# =========================
# 3. MOMENTUM: t-12 to t-2 (with continuity check)
# =========================
monthly = monthly.sort_values(["permno", "date"]).copy()

monthly["month_num"] = monthly["date"].dt.year * 12 + monthly["date"].dt.month
monthly["month_diff"] = monthly.groupby("permno")["month_num"].diff()

def count_consecutive(group):
    consecutive = pd.Series(index=group.index, dtype=float)
    count = 0
    for idx in group.index:
        if pd.isna(group.loc[idx, "month_diff"]):
            count = 1
        elif group.loc[idx, "month_diff"] == 1:
            count += 1
        else:
            count = 1
        consecutive.loc[idx] = count
    return consecutive

monthly["consecutive"] = (
    monthly.groupby("permno", group_keys=False)
    .apply(count_consecutive)
)

monthly["mom_raw"] = (
    monthly.groupby("permno")["ret"]
    .transform(lambda x: (1 + x).shift(2).rolling(11).apply(np.prod, raw=True) - 1)
)

monthly["mom"] = np.where(monthly["consecutive"] >= 13, monthly["mom_raw"], np.nan)

monthly = monthly.drop(columns=["month_num", "month_diff", "consecutive", "mom_raw"])

# =========================
# 4. S&P 500 BENCHMARK (value-weighted with start-of-month cap)
# =========================
def compute_sp500_benchmark(monthly_data):
    results = []

    for dt in monthly_data["date"].drop_duplicates().sort_values():
        month_slice = monthly_data.loc[
            monthly_data["date"] == dt, ["ret", "me_start"]
        ].dropna()

        total_me = month_slice["me_start"].sum()
        if total_me > 0:
            mkt_ret = (month_slice["ret"] * month_slice["me_start"] / total_me).sum()
            results.append({"date": dt, "sp500_return": mkt_ret})

    bench = pd.DataFrame(results).sort_values("date").reset_index(drop=True)
    bench["sp500_cum_return"] = (1 + bench["sp500_return"]).cumprod() - 1
    return bench

benchmark = compute_sp500_benchmark(monthly)

# =========================
# 5. STRATEGY FUNCTION
# trade_cost_bps = one-way cost per trade in bps
# =========================
def run_strategy(monthly_data, rebalance_type, trade_cost_bps=10, silent=False):
    one_way_cost = trade_cost_bps / 10000.0

    dates = monthly_data["date"].drop_duplicates().sort_values()

    if rebalance_type == "1M":
        rebalance_dates = dates
    elif rebalance_type == "3M":
        rebalance_dates = dates[dates.dt.month.isin([3, 6, 9, 12])]
    elif rebalance_type == "12M":
        rebalance_dates = dates[dates.dt.month == 12]
    else:
        raise ValueError("rebalance_type must be '1M', '3M', or '12M'")

    results = []
    prev_weights = {}
    total_exits = 0
    total_periods = 0

    for i in range(len(rebalance_dates) - 1):
        t = rebalance_dates.iloc[i]
        t_next = rebalance_dates.iloc[i + 1]

        cross = monthly_data.loc[
            monthly_data["date"] == t
        ].dropna(subset=["mom", "me_start"]).copy()

        if len(cross) < 20:
            continue

        total_periods += 1

        n = max(1, int(np.floor(len(cross) * 0.10)))
        selected = cross.sort_values("mom", ascending=False).head(n).copy()
        selected_permnos = set(selected["permno"])

        total_me = selected["me_start"].sum()
        if total_me <= 0:
            continue

        new_weights = dict(zip(selected["permno"], selected["me_start"] / total_me))

        # Standard turnover definition: 0.5 * sum |delta weights|
        all_permnos = set(new_weights.keys()) | set(prev_weights.keys())
        turnover = sum(
            abs(new_weights.get(p, 0.0) - prev_weights.get(p, 0.0))
            for p in all_permnos
        ) / 2.0

        # Since costs are paid on both sells and buys, total cost = 2 * turnover * one-way cost
        rebalance_tc = 2.0 * turnover * one_way_cost

        holding_months = sorted(
            monthly_data.loc[
                (monthly_data["date"] > t) & (monthly_data["date"] <= t_next), "date"
            ].unique()
        )

        current_permnos = set(selected_permnos)
        period_results = []
        total_exit_tc = 0.0
        period_exits = 0

        for j, dt in enumerate(holding_months):
            month_slice = monthly_data.loc[
                (monthly_data["date"] == dt) &
                (monthly_data["permno"].isin(current_permnos))
            ].copy()

            present_permnos = set(month_slice["permno"])
            exited_permnos = current_permnos - present_permnos
            period_exits += len(exited_permnos)

            # Forced exits due to missing future observations
            if exited_permnos:
                if j == 0:
                    # If exits occur immediately, use current target weights
                    exit_weight = sum(new_weights.get(p, 0.0) for p in exited_permnos)
                    total_exit_tc += exit_weight * one_way_cost
                else:
                    prev_month_dt = holding_months[j - 1]
                    prev_month_data = monthly_data.loc[
                        (monthly_data["date"] == prev_month_dt) &
                        (monthly_data["permno"].isin(current_permnos))
                    ].copy()

                    if not prev_month_data.empty:
                        prev_total_me = prev_month_data["me_end"].sum()
                        if prev_total_me > 0:
                            for p in exited_permnos:
                                p_data = prev_month_data.loc[prev_month_data["permno"] == p]
                                if not p_data.empty:
                                    exit_weight = p_data["me_end"].iloc[0] / prev_total_me
                                    total_exit_tc += exit_weight * one_way_cost

            current_permnos = present_permnos

            total_me_month = month_slice["me_start"].sum()
            if total_me_month > 0:
                month_slice["weight"] = month_slice["me_start"] / total_me_month
                port_ret = (month_slice["ret"] * month_slice["weight"]).sum()
                period_results.append({
                    "date": dt,
                    "portfolio_return": port_ret
                })

        total_exits += period_exits

        if period_results:
            port = pd.DataFrame(period_results)

            # Apply rebalance cost to first holding month
            port.loc[port.index[0], "portfolio_return"] -= rebalance_tc

            # Apply forced-exit cost to last observed month
            if total_exit_tc > 0:
                port.loc[port.index[-1], "portfolio_return"] -= total_exit_tc

            results.append(port)

        # Update portfolio weights for next rebalance date
        if period_results:
            last_dt = holding_months[-1] if len(holding_months) > 0 else t_next
            last_month_data = monthly_data.loc[
                (monthly_data["date"] == last_dt) &
                (monthly_data["permno"].isin(current_permnos))
            ].copy()

            if not last_month_data.empty:
                total_me_end = last_month_data["me_end"].sum()
                if total_me_end > 0:
                    prev_weights = dict(
                        zip(
                            last_month_data["permno"],
                            last_month_data["me_end"] / total_me_end
                        )
                    )
                else:
                    prev_weights = {}
            else:
                prev_weights = {}
        else:
            prev_weights = new_weights

    if not silent:
        print(f"\n=== EXIT SUMMARY ({rebalance_type} rebalancing) ===")
        print(f"Total holding periods: {total_periods}")
        print(f"Total forced exits: {total_exits}")
        if total_periods > 0:
            print(f"Average exits per period: {total_exits / total_periods:.2f}")

    if len(results) == 0:
        return pd.DataFrame(columns=["date", "portfolio_return", "cum_return"])

    results = pd.concat(results, ignore_index=True).sort_values("date")
    results["cum_return"] = (1 + results["portfolio_return"]).cumprod() - 1

    return results

# =========================
# 6. SUMMARY FUNCTION
# =========================
def summarize_results(results, label):
    if results.empty:
        return {
            "rebalancing": label,
            "n_months": 0,
            "total_return": np.nan,
            "annualized_return": np.nan,
            "annualized_volatility": np.nan,
            "sharpe": np.nan,
            "max_drawdown": np.nan
        }

    wealth = (1 + results["portfolio_return"]).cumprod()
    total_return = wealth.iloc[-1] - 1
    n_months = len(results)

    annualized_return = wealth.iloc[-1] ** (12 / n_months) - 1
    annualized_volatility = results["portfolio_return"].std(ddof=1) * np.sqrt(12)

    rf_annualized_average = 3.738 / 100
    sharpe = (
        (annualized_return - rf_annualized_average) / annualized_volatility
        if annualized_volatility > 0 else np.nan
    )

    max_drawdown = (wealth / wealth.cummax() - 1).min()

    return {
        "rebalancing": label,
        "n_months": n_months,
        "total_return": total_return,
        "annualized_return": annualized_return,
        "annualized_volatility": annualized_volatility,
        "sharpe": sharpe,
        "max_drawdown": max_drawdown
    }

# =========================
# 7. MOMENTUM PREMIUM FUNCTION
# =========================
def compute_momentum_premium(strategy_results, benchmark_common, label):
    merged = strategy_results.merge(
        benchmark_common[["date", "sp500_return"]],
        on="date",
        how="inner"
    )

    n = len(merged)
    if n == 0:
        return {
            "rebalancing": label,
            "premium_total": np.nan,
            "premium_annualized": np.nan
        }

    total_portfolio = (1 + merged["portfolio_return"]).cumprod().iloc[-1] - 1
    total_benchmark = (1 + merged["sp500_return"]).cumprod().iloc[-1] - 1

    premium_total = total_portfolio - total_benchmark
    premium_annualized = (
        (1 + total_portfolio) ** (12 / n) - 1
        - ((1 + total_benchmark) ** (12 / n) - 1)
    )

    return {
        "rebalancing": label,
        "premium_total": premium_total,
        "premium_annualized": premium_annualized
    }

# =========================
# 8. RUN FOR EACH COST LEVEL
# 15, 25, 30 = one-way per-trade costs in bps
# =========================
trade_cost_levels = [15, 25, 30]

for trade_cost_bps in trade_cost_levels:
    print(f"\n{'#' * 70}")
    print(f"# TRANSACTION COST PER TRADE (ONE-WAY): {trade_cost_bps} bps")
    print(f"{'#' * 70}")

    res_1m = run_strategy(monthly, "1M", trade_cost_bps=trade_cost_bps)
    res_3m = run_strategy(monthly, "3M", trade_cost_bps=trade_cost_bps)
    res_12m = run_strategy(monthly, "12M", trade_cost_bps=trade_cost_bps)

    # Align to common period
    start_date = max(
        res_1m["date"].min(),
        res_3m["date"].min(),
        res_12m["date"].min()
    )
    end_date = min(
        res_1m["date"].max(),
        res_3m["date"].max(),
        res_12m["date"].max()
    )

    res_1m = res_1m.loc[
        (res_1m["date"] >= start_date) & (res_1m["date"] <= end_date)
    ].copy()
    res_3m = res_3m.loc[
        (res_3m["date"] >= start_date) & (res_3m["date"] <= end_date)
    ].copy()
    res_12m = res_12m.loc[
        (res_12m["date"] >= start_date) & (res_12m["date"] <= end_date)
    ].copy()

    res_1m["cum_return"] = (1 + res_1m["portfolio_return"]).cumprod() - 1
    res_3m["cum_return"] = (1 + res_3m["portfolio_return"]).cumprod() - 1
    res_12m["cum_return"] = (1 + res_12m["portfolio_return"]).cumprod() - 1

    bench_common = benchmark.loc[
        (benchmark["date"] >= start_date) & (benchmark["date"] <= end_date)
    ].copy()
    bench_common["sp500_cum_return"] = (1 + bench_common["sp500_return"]).cumprod() - 1

    # Summary
    summary = pd.DataFrame([
        summarize_results(res_1m, "1 month"),
        summarize_results(res_3m, "3 months"),
        summarize_results(res_12m, "12 months")
    ])

    premium_summary = pd.DataFrame([
        compute_momentum_premium(res_1m, bench_common, "1 month"),
        compute_momentum_premium(res_3m, bench_common, "3 months"),
        compute_momentum_premium(res_12m, bench_common, "12 months")
    ])

    # Print
    print(f"\n=== COMMON PERIOD: {start_date.strftime('%Y-%m')} to {end_date.strftime('%Y-%m')} ===\n")

    summary_display = summary.copy()
    for col in ["total_return", "annualized_return", "annualized_volatility", "sharpe", "max_drawdown"]:
        summary_display[col] = summary_display[col].round(4)

    print(f"=== STRATEGY PERFORMANCE (trade cost: {trade_cost_bps} bps one-way) ===")
    print(summary_display.to_string(index=False))
    print()

    sp500_total = bench_common["sp500_cum_return"].iloc[-1]
    sp500_n = len(bench_common)
    sp500_ann = (1 + sp500_total) ** (12 / sp500_n) - 1

    print("=== S&P 500 BENCHMARK (common period) ===")
    print(f"Total return: {sp500_total:.4f}")
    print(f"Annualized return: {sp500_ann:.4f}")
    print()

    premium_display = premium_summary.copy()
    for col in ["premium_total", "premium_annualized"]:
        premium_display[col] = premium_display[col].round(4)

    print("=== MOMENTUM PREMIUM (strategy - S&P 500) ===")
    print(premium_display.to_string(index=False))

/tmp/ipykernel_28849/1339590065.py:102: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(count_consecutive)



######################################################################
# TRANSACTION COST PER TRADE (ONE-WAY): 15 bps
######################################################################

=== EXIT SUMMARY (1M rebalancing) ===
Total holding periods: 204
Total forced exits: 70
Average exits per period: 0.34

=== EXIT SUMMARY (3M rebalancing) ===
Total holding periods: 68
Total forced exits: 62
Average exits per period: 0.91

=== EXIT SUMMARY (12M rebalancing) ===
Total holding periods: 17
Total forced exits: 43
Average exits per period: 2.53

=== COMMON PERIOD: 1992-01 to 2008-12 ===

=== STRATEGY PERFORMANCE (trade cost: 15 bps one-way) ===
rebalancing  n_months  total_return  annualized_return  annualized_volatility  sharpe  max_drawdown
    1 month       204        7.2084             0.1318                 0.1956  0.4828       -0.4263
   3 months       204        6.3693             0.1247                 0.2028  0.4305       -0.6110
  12 months       204        4.5250             0

In [ ]:
def print_combined_spread_table(monthly, benchmark, trade_cost_levels=[15, 25, 30]):
    all_results = {}

    for trade_cost_bps in trade_cost_levels:
        res_1m = run_strategy(monthly, "1M", trade_cost_bps=trade_cost_bps)
        res_3m = run_strategy(monthly, "3M", trade_cost_bps=trade_cost_bps)
        res_12m = run_strategy(monthly, "12M", trade_cost_bps=trade_cost_bps)

        start_date = max(
            res_1m["date"].min(),
            res_3m["date"].min(),
            res_12m["date"].min()
        )
        end_date = min(
            res_1m["date"].max(),
            res_3m["date"].max(),
            res_12m["date"].max()
        )

        res_1m = res_1m[(res_1m["date"] >= start_date) & (res_1m["date"] <= end_date)].copy()
        res_3m = res_3m[(res_3m["date"] >= start_date) & (res_3m["date"] <= end_date)].copy()
        res_12m = res_12m[(res_12m["date"] >= start_date) & (res_12m["date"] <= end_date)].copy()

        res_1m["cum_return"] = (1 + res_1m["portfolio_return"]).cumprod() - 1
        res_3m["cum_return"] = (1 + res_3m["portfolio_return"]).cumprod() - 1
        res_12m["cum_return"] = (1 + res_12m["portfolio_return"]).cumprod() - 1

        all_results[trade_cost_bps] = {
            "1M": summarize_results(res_1m, "1 month"),
            "3M": summarize_results(res_3m, "3 months"),
            "12M": summarize_results(res_12m, "12 months")
        }

    w = 8
    line = "=" * 100

    print(line)
    print("  STRATEGY PERFORMANCE - TRANSACTION COST SENSITIVITY")
    print(line)

    # Header row 1
    print(f"  {'':14s}", end="")
    for c in trade_cost_levels:
        block = f"{c} bps"
        print(f" |{block:^{3*w}s}", end="")
    print(" |")

    # Header row 2
    print(f"  {'':14s}", end="")
    for _ in range(len(trade_cost_levels)):
        print(f" |{'1M':>{w}s}{'3M':>{w}s}{'12M':>{w}s}", end="")
    print(" |")
    print("-" * 100)

    metrics = [
        ("Ann. Return", "annualized_return", True),
        ("Ann. Vol", "annualized_volatility", True),
        ("Sharpe", "sharpe", False),
        ("Max DD", "max_drawdown", True),
    ]

    for metric_name, metric_key, is_pct in metrics:
        print(f"  {metric_name:14s}", end="")
        for trade_cost_bps in trade_cost_levels:
            print(" |", end="")
            for rebal in ["1M", "3M", "12M"]:
                val = all_results[trade_cost_bps][rebal][metric_key]

                if pd.isna(val):
                    text = "NA"
                    print(f"{text:>{w}s}", end="")
                elif is_pct:
                    print(f"{val*100:>{w-1}.2f}%", end="")
                else:
                    print(f"{val:>{w}.4f}", end="")
        print(" |")

    print(line)
print_combined_spread_table(monthly, benchmark, trade_cost_levels=[15, 25, 30])


=== EXIT SUMMARY (1M rebalancing) ===
Total holding periods: 204
Total forced exits: 70
Average exits per period: 0.34

=== EXIT SUMMARY (3M rebalancing) ===
Total holding periods: 68
Total forced exits: 62
Average exits per period: 0.91

=== EXIT SUMMARY (12M rebalancing) ===
Total holding periods: 17
Total forced exits: 43
Average exits per period: 2.53

=== EXIT SUMMARY (1M rebalancing) ===
Total holding periods: 204
Total forced exits: 70
Average exits per period: 0.34

=== EXIT SUMMARY (3M rebalancing) ===
Total holding periods: 68
Total forced exits: 62
Average exits per period: 0.91

=== EXIT SUMMARY (12M rebalancing) ===
Total holding periods: 17
Total forced exits: 43
Average exits per period: 2.53

=== EXIT SUMMARY (1M rebalancing) ===
Total holding periods: 204
Total forced exits: 70
Average exits per period: 0.34

=== EXIT SUMMARY (3M rebalancing) ===
Total holding periods: 68
Total forced exits: 62
Average exits per period: 0.91

=== EXIT SUMMARY (12M rebalancing) ===
Tot